# Biomni Eval1: Dataset inspection, Agent Run, CSV export

## Objectives
1. Inspect teh Biomni Eval1 dataset
2. Sample 30 random queries
3. Run the Biomni agent with llama3
4. Evaluate the outputs
5. Export the results to CSV
6. Reload the CSV for comparison and analysis

The final csv will include the following main collumns:
- 'input_query'
- 'dataset_eval1_answer'
- 'agent_reasoning_trace'
- 'agent_output_answer'

#### Imports

In [1]:
from __future__ import annotations

import csv
import json
import random
import time
from pathlib import Path
from typing import Any

import pandas as pd
from datasets import load_dataset
from rich.console import Console
from rich.table import Table

from bio_agents.frameworks import FRAMEWORK_REGISTRY
from bio_agents.tasks.biomni_eval1.task import BiomniEval1Task

console = Console()

#### Auxiliary Functions

In [2]:
def _safe_json(value: Any) -> str:
    try:
        return json.dumps(value, ensure_ascii=False)
    except Exception:
        return str(value)

#### Load and inspect the Biomni Eval1 dataset

This section loads the test split of the Eval1 benchmark and inspects:

- number of rows
- column names
- example entries

This helps confirm which dataset fields should be used later in the CSV export.

In [3]:
console.print("[bold]Loading biomni/Eval1 dataset...[/bold]")
dataset = load_dataset("biomni/Eval1", split="test")
rows = list(dataset)

print(f"Number of rows: {len(rows)}")
print("Columns:")
print(dataset.column_names)

Loading biomni/Eval1 dataset...

Number of rows: 433
Columns:
['instance_id', 'task_instance_id', 'prompt', 'task_name', 'split', 'answer']


In [4]:
# Inspect the first example to understand the structure of the data
example = dataset[0]
example

{'instance_id': 0,
 'task_instance_id': 134,
 'prompt': 'Your task is to identify the most promising variant associated wtih a given GWAS phenotype for futher examination. \nFrom the list, prioritize the top associated variant (matching one of the given variant). \nGWAS phenotype: Bradykinin\nVariants: rs7700133, rs1280, rs7651090, rs4253311, rs3738934, rs7385804, rs1367117, rs4808136, rs10087900, rs855791, rs12678919\n',
 'task_name': 'gwas_variant_prioritization',
 'split': 'val',
 'answer': 'rs4253311'}

From the inspected example, the key fields used in this notebook are:

- `prompt` → input query
- `answer` → dataset reference answer
- `task_name`
- `task_instance_id`

At this stage, the dataset reference answer is treated as the expected answer for comparison.

In [5]:
df = pd.DataFrame(dataset)
df.head()

,instance_id,task_instance_id,prompt,task_name,split,answer
0,0,134,Your task is to identify the most promising va...,gwas_variant_prioritization,val,rs4253311
1,1,178,Your task is to identify the most promising va...,gwas_variant_prioritization,val,rs9369898
2,2,197,Your task is to identify the most promising va...,gwas_variant_prioritization,val,rs7542172
3,3,63,Your task is to identify the most promising va...,gwas_variant_prioritization,val,rs1801725
4,4,54,Your task is to identify the most promising va...,gwas_variant_prioritization,val,rs4343


#### Configuration

In [6]:
sample_size = 30
seed = 42
framework = "biomni"
model = "llama3"
commercial_mode = False
timeout_seconds = 600

output_path = Path("results/biomni_eval1_sample/sample_30_seed42.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print("sample_size =", sample_size)
print("seed =", seed)
print("framework =", framework)
print("model =", model)
print("output_path =", output_path)

Configuration:
sample_size = 30
seed = 42
framework = biomni
model = llama3
output_path = results\biomni_eval1_sample\sample_30_seed42.csv


In [7]:
rng = random.Random(seed)
sampled_rows = rng.sample(rows, sample_size)

table = Table(title=f"Biomni Eval1 random sample (n={sample_size}, seed={seed})")
table.add_column("idx", style="cyan")
table.add_column("task_name", style="magenta")
table.add_column("instance_id", style="yellow")
table.add_column("prompt_preview", style="green")

for i, row in enumerate(sampled_rows, start=1):
    prompt = row.get("prompt", "")
    preview = str(prompt).replace("\n", " ")
    if len(preview) > 90:
        preview = preview[:87] + "..."
    table.add_row(
        str(i),
        str(row.get("task_name", "")),
        str(row.get("task_instance_id", row.get("instance_id", ""))),
        preview,
    )

console.print(table)

                                    Biomni Eval1 random sample (n=30, seed=42)                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ idx ┃ task_name                       ┃ instance_id ┃ prompt_preview                                            ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ lab_bench_dbqa                  │ 387         │ The following is a multiple choice question about         │
│     │                                 │             │ biology. Please answer by responding ...                  │
│ 2   │ screen_gene_retrieval           │ 259         │ Your task is to identify the gene with the strongest      │
│     │                                 │             │ perturbation effect for the follow...                     │
│ 3   │ gwas_variant_prioritization     │ 88          │ Your task is to identify the most promising variant       │
│     │                                 │             │ associated wtih a given GWAS phenot...                    │
│ 4   │ gwas_causal_gene_opentargets    │ 27          │ Your task is to identify likely causal genes within a     │
│     │                                 │             │ locus for a given GWAS phenotype....                      │
│ 5   │ lab_bench_seqqa                 │ 187         │ The following is a multiple choice question about         │
│     │                                 │             │ biology. Please answer by responding ...                  │
│ 6   │ lab_bench_seqqa                 │ 418         │ The following is a multiple choice question about         │
│     │                                 │             │ biology. Please answer by responding ...                  │
│ 7   │ lab_bench_seqqa                 │ 327         │ The following is a multiple choice question about         │
│     │                                 │             │ biology. Please answer by responding ...                  │
│ 8   │ screen_gene_retrieval           │ 88          │ Your task is to identify the gene with the strongest      │
│     │                                 │             │ perturbation effect for the follow...                     │
│ 9   │ gwas_causal_gene_opentargets    │ 200         │ Your task is to identify likely causal genes within a     │
│     │                                 │             │ locus for a given GWAS phenotype....                      │
│ 10  │ screen_gene_retrieval           │ 201         │ Your task is to identify the gene with the strongest      │
│     │                                 │             │ perturbation effect for the follow...                     │
│ 11  │ gwas_causal_gene_opentargets    │ 540         │ Your task is to identify likely causal genes within a     │
│     │                                 │             │ locus for a given GWAS phenotype....                      │
│ 12  │ crispr_delivery                 │ 42          │ Given the case description, identify the MOST relevant    │
│     │                                 │             │ CRISPR delivery method from the ...                       │
│ 13  │ screen_gene_retrieval           │ 236         │ Your task is to identify the gene with the strongest      │
│     │                                 │             │ perturbation effect for the follow...                     │
│ 14  │ lab_bench_dbqa                  │ 53          │ The following is a multiple choice question about         │
│     │                                 │             │ biology. Please answer by responding ...                  │
│ 15  │ gwas_causal_gene_pharmaprojects │ 1158        │ Your task is to identify likely causal genes within a     │
│     │                                 │             │ locus for a given GWAS phenotype....                      │
│ 16  │ gwas_variant_prioritization     │ 207         │ 

#### Run the agent e collect the results

In [ ]:
runner = FRAMEWORK_REGISTRY[framework]()
results_for_csv = []

console.print(
    f"[bold]Running {sample_size} sampled Eval1 queries with framework={framework}, model={model}[/bold]"
)

for i, row in enumerate(sampled_rows, start=1):
    task_name = row.get("task_name", "")
    task_instance_id = row.get("task_instance_id", row.get("instance_id", ""))
    prompt = row.get("prompt", "")
    dataset_eval1_answer = row.get("answer", "")

    task = BiomniEval1Task(
        task_name=task_name,
        task_instance_id=task_instance_id,
        prompt=prompt,
    )

    task_input = task.get_input()
    tools = task.get_tools()

    console.print(
        f"[cyan][{i}/{sample_size}][/cyan] "
        f"{task_name} | instance_id={task_instance_id}"
    )

    start = time.perf_counter()

    try:
        agent_result = runner.run(
            task_input.prompt,
            tools,
            model,
            commercial_mode=commercial_mode,
            timeout_seconds=timeout_seconds,
        )
        duration_s = round(time.perf_counter() - start, 3)

        eval_result = task.evaluate(agent_result)

        # Observable reasoning trace based on tool calls
        agent_reasoning_trace = _safe_json(agent_result.tool_calls)

        results_for_csv.append(
            {
                "sample_index": i,
                "seed": seed,
                "framework": framework,
                "model": model,
                "task_name": task_name,
                "task_instance_id": task_instance_id,
                "input_query": prompt,
                "dataset_eval1_answer": dataset_eval1_answer,
                "agent_reasoning_trace": agent_reasoning_trace,
                "agent_output_answer": agent_result.output,
                "score": eval_result.score,
                "passed": eval_result.passed,
                "duration_s": duration_s,
                "tool_calls_count": len(agent_result.tool_calls),
                "agent_metadata": _safe_json(agent_result.metadata),
                "metrics": _safe_json(eval_result.metrics),
                "eval_error": eval_result.metrics.get("eval_error", "")
                if isinstance(eval_result.metrics, dict)
                else "",
            }
        )

    except Exception as exc:
        duration_s = round(time.perf_counter() - start, 3)

        results_for_csv.append(
            {
                "sample_index": i,
                "seed": seed,
                "framework": framework,
                "model": model,
                "task_name": task_name,
                "task_instance_id": task_instance_id,
                "input_query": prompt,
                "dataset_eval1_answer": dataset_eval1_answer,
                "agent_reasoning_trace": "",
                "agent_output_answer": "",
                "score": 0.0,
                "passed": False,
                "duration_s": duration_s,
                "tool_calls_count": 0,
                "agent_metadata": "{}",
                "metrics": "{}",
                "eval_error": str(exc),
            }
        )

Running 30 sampled Eval1 queries with framework=biomni, model=llama3

[1/30] lab_bench_dbqa | instance_id=387

🎓 Academic mode: Using all datasets (including non-commercial)

🔧 BIOMNI CONFIGURATION
📋 DEFAULT CONFIG (Including Database LLM):
  Path: ./data
  Timeout Seconds: 600
  Llm: claude-sonnet-4-5
  Temperature: 0.7
  Use Tool Retriever: True
  Commercial Mode: Academic (all datasets)

🤖 AGENT LLM (Constructor Override):
  LLM Model: llama3
  Base URL: http://localhost:11434/v1

Checking and downloading missing data lake files...
Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

The following is a multiple choice question about biology.
Please answer by responding with the letter of the correct answer.

Question: According to ClinVar, which of the following sequences is most likely to be benign? 
Options:
A.MADEKTFRIGFIVLGLFLLALGTFLMSHDRPQVYGTFYAMGSVMVIGGIIWSMCQCYPKITFVPADSDFQGILSPKAMGLLENGLAAEMKSPSPQPPYVRLWEEAAYDQSLPDFSHIQMKVMSYSEDHRSLLAPEMGQPKLGTSDGGEGGPGDVQAWMEAAVVIHKGSDESEGERRLTQSWPGPLACPQGPAPLASFQDDLDMDS

#### Export results to CSV

In [ ]:
fieldnames = [
    "sample_index",
    "seed",
    "framework",
    "model",
    "task_name",
    "task_instance_id",
    "input_query",
    "dataset_eval1_answer",
    "agent_reasoning_trace",
    "agent_output_answer",
    "score",
    "passed",
    "duration_s",
    "tool_calls_count",
    "agent_metadata",
    "metrics",
    "eval_error",
]

with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results_for_csv)

passed_count = sum(1 for r in results_for_csv if r["passed"])
avg_score = sum(float(r["score"]) for r in results_for_csv) / len(results_for_csv)

print(f"CSV saved to: {output_path}")
print(f"Passed: {passed_count}/{len(results_for_csv)}")
print(f"Average score: {avg_score:.3f}")

In [ ]:
results_df = pd.read_csv(output_path)
results_df.head()